# 24 - Train Conditional Value Diffusion Offline

This uses the FrozenLake replay and tokenization from [02 - Offline DQN](02_train_offline_dqn.ipynb).
The transformer supplies only its final value-token embedding. A separate
`ValueFlow` RMS-normalizes that context and predicts velocity
from the context, current return sample, and explicit flow time.

The training loop draws Gaussian noise; `ValueFlow` integrates its trajectory. `DIFFUSION_STEPS` is
independent of transformer depth. `ValueDiffusionObjective` trains velocities
against coupled Bellman targets using the [paper's Q-only formulation](https://arxiv.org/html/2611.02160v1#A4.SS5)
and discrete greedy action selection. Training requires CUDA and example 2's dataset.


In [ ]:
import json
from collections.abc import Iterator
from dataclasses import dataclass
from copy import deepcopy
from pathlib import Path

import torch
from torch import nn

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    Augmenter,
    DataLoader,
    Tokenizer,
    compose,
    load_stores_from_hub,
    save_tokenizer,
)
from mouse_core.models import (
    Backbone,
    DecodeCache,
    prepare_decode,
    Embedding,
    Model,
    MultiEmbedding,
    MLP,
    save_model,
)
from mouse_core.objectives import (
    ValueDiffusionObjective,
    boundary_discount,
)

# Data: same FrozenLake replay as example 2.
DATASET_ID = "mouse-example-frozenlake-dataset"
MAX_ACTIONS = 4
MAX_OBS_DISCRETE = 64
MAX_STEPS_PER_EPISODE = 24
REPEAT_COUNT = 20

# Model
PRETRAINED = "Qwen/Qwen3-0.6B"
DIFFUSION_HIDDEN_DIM = 256
DIFFUSION_NETWORK_LAYERS = 2
DIFFUSION_OUTPUT_SCALE = 0.1
DIFFUSION_STEPS = 8
COMPUTE_TOKENS = 8
DIFFUSION_RETURN_NUM_FREQUENCIES = 16
DIFFUSION_TIME_NUM_FREQUENCIES = 16

# Training: context is encoded once per model.
TOKEN_BUDGET = 1024
NUM_CYCLES = 2
TRAIN_STEPS = 50

# Objective: same base discount as example 2; the noise level scales rewards on ALL steps.
GAMMA_STEP = 1.0

# Optimizer learning rates
LR_EMBEDDING = 1e-5
LR_BACKBONE = 1e-5
LR_DIFFUSION = 1e-5

# Delayed model interpolation rates
POLYAK_TAU_EMBEDDING = 0.1
POLYAK_TAU_BACKBONE = 0.1
POLYAK_TAU_DIFFUSION = 0.001

# Save locally after training.
OUTPUT_DIR = Path("outputs/offline_value_diffusion")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("This notebook uses Mouse FlexAttention training and requires CUDA.")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved
`Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train", force_download=True)


## Data Pipeline

`DataLoader` samples complete outer tasks with
`sample_ranges=sample_ranges`, keeping all repeated lake episodes together.
Each sampled task gets a unique sequence ID so attention and objective transitions
cannot cross between tasks.

Pass exactly one of `token_budget` or `events_budget`. Each batch contains whole samples
up to that budget. An oversized first sample raises. Token budgets count packed tokens,
including episode prompts; event budgets count stored source events, not decision
readouts.

The pipeline is
`source events → sparse decisions → augmenter → tokenizer → pack → embedding`:

| Stage       | Role                                                                                                             |
| ----------- | ---------------------------------------------------------------------------------------------------------------- |
| `Augmenter` | Permutes actions and observations consistently within a sampled task; action permutations also update `_q_star`. |
| `Tokenizer` | Tokenizes matching fields per event, groups tokens by `step_index`, and retains explicit objective labels.       |

`train_transform = compose(stages=(augmenter.events, tokenizer.encode))` is passed as
the loader’s `transform`. `next_batch()` returns `(inputs, objective_data)` as
TensorDicts; pass `objective_data["sequence_ids"]` as the objective’s `sequence_ids`.
[Inference](15_inference.ipynb) uses the saved tokenizer without augmentation so
selected actions match the environment.

For per-event encoding, configure `Tokenizer` with a flat `input_fields` list,
`step_field="step_index"`, and `episode_field="episode_index"`, then pass
`transform=tokenizer.encode` directly to the loader. It accumulates event tokens until
the step marker changes. Define a local range function to select windows of whole steps
N whole steps within an episode. Augmentation for this path must transform event
payloads before encoding; `augmenter.events` preserves source events and shares
episode-keyed draws.

`COMPUTE_TOKENS` sets a constant number of compute tokens before every value readout.
Set it to zero to disable compute text. The field filters on `action_requested`
and uses the literal format `" ." * COMPUTE_TOKENS`. No callback or runtime context
is needed. The configured text is saved with the tokenizer and reused at inference.

Qwen encodes each spaced period as one token (ID 659); ten bare periods can merge
into just one token. Passive and
terminal events emit no compute text. Compute tokens count toward the batch budget.

Only the newline has `head_output=True`. No noise is generated or stored by
the tokenizer. Both encoders receive the same ordinary packed input batch.
Gaussian sampling takes place in `ValueFlow` after the transformer forward.


In [ ]:
def action_name(action):
    return ("a", "b", "c", "d")[action]


episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\nx marks no action at the beginning of an episode.\n"
)

augmenter = Augmenter(
    seed_field="episode_index",
    fields=[
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "action",
            "input_vector_field": "_q_star",
            "vocab_size": MAX_ACTIONS,
        },
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
        },
    ],
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": 'episode {repeat_episode_index+1}\n',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "x",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "format": '{action_name(action)}',
            "format_functions": {"action_name": action_name},
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ',{observation}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ',r={reward:g}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": " ." * COMPUTE_TOKENS,
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": ["inner_terminated", "inner_truncated"],
            "format": "\n\n",
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "_terminated",
            "output_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "_truncated",
            "output_field": "truncated",
            "missing_value": False,
        },
    ],
)


train_transform = compose(stages=(augmenter.events, tokenizer.encode))


def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Train only on episodes with a recorded final environment result."""
    last = events[-1]
    return last["provider"] == "environment" and (
        last["payload"].get("_terminated") is True
        or last["payload"].get("_truncated") is True
    )


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    transform=train_transform,
    prefetch=4,
    num_workers=4,
)


## Build Encoder and Value Flow Model

The encoder returns the **final** transformer readout at each value token as
`[readouts, hidden_dim]`. It does not collect intermediate layer states.
Like Example 2, it also accepts `cache=` / `use_cache=True` for incremental
inference and returns the updated cache. Cached readouts have shape
`[batch, readouts, hidden_dim]`; flatten the first two dimensions before passing
them to the flow model. Offline training uses the ordinary packed forward.

The `ValueFlow` class defined below applies learned RMSNorm to that readout
and projects it into `DIFFUSION_HIDDEN_DIM`. It combines this fixed context with
independent Fourier projections of the current scalar return and flow time, then
predicts one velocity per action through an MLP. Context is normalized/projected
once and reused across all flow steps. No flow noise enters the transformer.
The complete implementation is here: scalar Fourier features, the velocity
network, Euler integration, and trajectory storage. `ValueFlow` inherits
directly from `nn.Module`.

The online encoder and complete ValueFlow model train jointly. Frozen copies
of both supply delayed targets, including greedy action selection.


In [ ]:
class Encoder(Model):
    def __init__(self, *, pretrained, tokenizer_sources, dtype):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            },
        )
        self.backbone = Backbone(pretrained=pretrained, dtype=dtype)

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        head_token_ids = tensors["head_mask"].nonzero(as_tuple=True)[0]
        # The ValueFlow model RMS-normalizes this final readout once.
        # Return samples and flow time never enter the transformer.
        embed = self.embedding(
            tensors["token_ids"],
            numeric_values=tensors["numeric_values"],
            numeric_mask=tensors["numeric_mask"],
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
            position_values=tensors["position_values"],
            position_mask=tensors["position_mask"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=tensors["sequence_ids"],
                head_mask=tensors["head_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(self.backbone.decode_session(batch_size=tokens["num_sequences"]),)
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape, features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=tensors["sequence_ids"])
            h = features[head_token_ids]
        return {"z": h, "cache": cache}


class ReturnFeatures(nn.Module):
    """Project a scalar flow input and its Fourier features into a diffusion feature vector.

    The raw scalar distinguishes inputs with identical periodic features.
    This projection neither samples noise nor modifies token embeddings.
    """

    def __init__(self, *, num_frequencies: int, hidden_dim: int):
        super().__init__()
        if type(num_frequencies) is not int:
            raise TypeError("num_frequencies must be an int")
        if num_frequencies < 1:
            raise ValueError("num_frequencies must be positive")
        count = num_frequencies
        self.register_buffer("bands", torch.pi * 2.0 ** torch.arange(count, dtype=torch.float32), persistent=False)
        self.proj = nn.Linear(1 + 2 * count, hidden_dim, bias=False, dtype=torch.float32)
        nn.init.normal_(self.proj.weight, std=(count + 1) ** -0.5)

    def forward(self, values):
        raw = values.to(self.proj.weight.dtype)[:, None]
        angles = raw * self.bands
        return self.proj(torch.cat((raw, angles.sin(), angles.cos()), dim=-1))


@dataclass(frozen=True)
class FlowTrajectory:
    """Inputs and velocities [P,T,A], endpoint returns [P,A], and times [T]."""

    samples: torch.Tensor
    velocities: torch.Tensor
    returns: torch.Tensor
    noise_levels: torch.Tensor


class ValueFlow(nn.Module):
    """Conditional velocity network and Euler flow integration."""

    noise_levels: torch.Tensor

    def __init__(self, *, context_dim, hidden_dim, num_actions, num_layers,
                 return_num_frequencies, time_num_frequencies, scale, num_steps,
                 noise_levels=None):
        super().__init__()
        if type(num_actions) is not int or num_actions < 1:
            raise ValueError("num_actions must be a positive integer")
        self.num_actions = num_actions
        if type(num_steps) is not int or num_steps < 1:
            raise ValueError("num_steps must be a positive integer")
        levels = (torch.arange(num_steps, dtype=torch.float32) / num_steps if noise_levels is None
                  else torch.as_tensor(noise_levels, dtype=torch.float32).detach().clone())
        if (levels.shape != (num_steps,)
                or not bool(torch.isfinite(levels).all()) or levels[0] != 0
                or not bool((levels[1:] > levels[:-1]).all()) or levels[-1] > 1):
            raise ValueError("noise_levels must have num_steps entries, start at zero, and increase strictly within [0,1]")
        self.register_buffer("noise_levels", levels)
        self.num_steps = num_steps
        for name, value in (("context_dim", context_dim), ("hidden_dim", hidden_dim),
                            ("num_actions", num_actions), ("num_layers", num_layers)):
            if type(value) is not int or value < 1:
                raise ValueError(f"{name} must be a positive integer")
        self.context_dim = context_dim
        self.hidden_dim = hidden_dim
        self.num_actions = num_actions
        self.context_norm = nn.RMSNorm(context_dim, eps=1e-5, dtype=torch.float32)
        self.context_projection = nn.Linear(context_dim, hidden_dim, dtype=torch.float32)
        self.return_features = ReturnFeatures(hidden_dim=hidden_dim, num_frequencies=return_num_frequencies)
        self.time_features = ReturnFeatures(hidden_dim=hidden_dim, num_frequencies=time_num_frequencies)
        self.input_projection = nn.Linear(3 * hidden_dim, hidden_dim, dtype=torch.float32)
        self.mlp = MLP(in_features=hidden_dim, out_features=num_actions, hidden_dim=hidden_dim,
                       num_layers=num_layers, scale=scale, normalize_input=False)

    def _context(self, features: torch.Tensor) -> torch.Tensor:
        if features.ndim != 2 or features.shape[-1] != self.context_dim:
            raise ValueError("Context must be final transformer readouts [P,context_dim]")
        with torch.autocast(device_type=features.device.type, enabled=False):
            return self.context_projection(self.context_norm(features.float()))

    def _predict(self, context: torch.Tensor, samples: torch.Tensor, times: torch.Tensor) -> torch.Tensor:
        with torch.autocast(device_type=context.device.type, enabled=False):
            shape = (*samples.shape, self.hidden_dim)
            context = context.reshape(samples.shape[0], *((1,) * (samples.ndim - 1)), self.hidden_dim).expand(shape)
            returns = self.return_features(samples.flatten()).reshape(shape)
            time = self.time_features(times).reshape(1, len(times), *((1,) * (samples.ndim - 2)), self.hidden_dim).expand(shape)
            inputs = torch.cat((context, returns, time), dim=-1)
            return self.mlp(torch.nn.functional.silu(self.input_projection(inputs)))

    def forward(self, features: torch.Tensor, samples: torch.Tensor, *, times: torch.Tensor) -> torch.Tensor:
        """Evaluate scalar inputs [P,T] or matched action trajectories [P,T,A]."""
        return self._velocity(self._context(features), samples, times)

    def _velocity(self, context: torch.Tensor, samples: torch.Tensor, times: torch.Tensor) -> torch.Tensor:
        if samples.ndim not in (2, 3) or samples.shape[0] != context.shape[0]:
            raise ValueError("Return inputs must be [P,T] or [P,T,A] matching the context rows")
        if samples.ndim == 3 and samples.shape[-1] != self.num_actions:
            raise ValueError("Action trajectories must match num_actions")
        if times.shape != (samples.shape[1],):
            raise ValueError("times must have one value per flow step")
        if samples.device != context.device or times.device != samples.device:
            raise ValueError("Context, return inputs, and times must share a device")
        if not bool((torch.isfinite(times) & (times >= 0) & (times <= 1)).all()):
            raise ValueError("times must be finite and within [0,1]")
        velocities = self._predict(context, samples, times)
        if velocities.shape != (*samples.shape, self.num_actions):
            raise ValueError("The velocity network must produce num_actions outputs per return input")
        return velocities if samples.ndim == 2 else velocities.diagonal(dim1=-2, dim2=-1)

    def _steps(self, context: torch.Tensor, initial_samples: torch.Tensor
               ) -> Iterator[tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
        """Yield each scalar input, velocity, and following sample without storing history."""
        if (initial_samples.ndim != 2 or initial_samples.shape[0] != context.shape[0]
                or initial_samples.shape[-1] != self.num_actions):
            raise ValueError("initial_samples must be [P,A] matching the context rows and num_actions")
        if initial_samples.dtype != torch.float32 or not bool(torch.isfinite(initial_samples).all()):
            raise ValueError("initial_samples must be finite float32 values")
        if initial_samples.device != context.device:
            raise ValueError("Context and initial_samples must share a device")
        levels = self.noise_levels.to(initial_samples.device)
        intervals = torch.cat((levels[1:], levels.new_ones(1))) - levels
        current = initial_samples
        for time, interval in zip(levels, intervals, strict=True):
            velocity = self._velocity(context, current[:, None], time.reshape(1)).squeeze(1)
            if velocity.shape != current.shape or velocity.device != current.device or velocity.dtype != torch.float32:
                raise ValueError("Velocity must be float32 matching the sample shape and device")
            following = current + interval * velocity
            yield current, velocity, following
            current = following

    def trajectory(self, features: torch.Tensor, *, initial_samples: torch.Tensor) -> FlowTrajectory:
        """Expand one seed per readout [P] across actions and integrate its trajectory."""
        if initial_samples.shape != (len(features),):
            raise ValueError("initial_samples must contain one seed per readout [P]")
        initial_samples = initial_samples[:, None].expand(-1, self.num_actions)
        context = self._context(features)
        samples, velocities = [], []
        returns = initial_samples
        for current, velocity, returns in self._steps(context, initial_samples):
            samples.append(current)
            velocities.append(velocity)
        return FlowTrajectory(torch.stack(samples, 1), torch.stack(velocities, 1),
                              returns, self.noise_levels.to(features.device))



encoder = Encoder(
    pretrained=PRETRAINED,
    tokenizer_sources=tokenizer.tokenizer_sources,
    dtype=torch.float32,
).train().to(device)

flow = ValueFlow(
    context_dim=encoder.backbone.hidden_dim,
    hidden_dim=DIFFUSION_HIDDEN_DIM,
    num_actions=MAX_ACTIONS,
    num_layers=DIFFUSION_NETWORK_LAYERS,
    scale=DIFFUSION_OUTPUT_SCALE,
    return_num_frequencies=DIFFUSION_RETURN_NUM_FREQUENCIES,
    time_num_frequencies=DIFFUSION_TIME_NUM_FREQUENCIES,
    num_steps=DIFFUSION_STEPS,
).train().to(device)

modules = nn.ModuleDict({"encoder": encoder, "flow": flow})
delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
delayed_flow = deepcopy(flow).requires_grad_(False).eval()

print(encoder)
print(flow)


## Train

The objective follows the DQN calling style: replay labels, sequence and readout
indices, online predictions, and delayed backup predictions go into the loss.
It handles action selection, terminal masking, automatic-step rewards, discounts,
Value Forcing targets, and loss weighting.

Unlike scalar DQN, the online velocity needs a Bellman-transformed input.
`objective.inputs(...)` constructs that tensor, including replay alignment,
action selection, terminal masking, rewards, discounts, and noise correction.
After obtaining z_tau from the delayed trajectory, the training loop explicitly
evaluates delayed Q at z_tau and online Q at the transformed input x_tau.
The delayed forward is a batched evaluation of the sampled path; the sampling
pass has already evaluated velocities during integration.
Both network evaluations finish before the objective receives their predictions.
The objective never evaluates either model or draws noise.

One initial Z draw is shared by all readouts in a replay sequence, including s
and s′. The independent correction noise N is drawn per readout. Both the input
and velocity target use the same delayed trajectory, greedy action, and N:

- `x_tau = tau*reward + discount*z_tau + (1-tau)*sqrt(1-discount**2)*N`
- `target = reward + discount*v_next(z_tau) - sqrt(1-discount**2)*N`

The online velocity is evaluated at x_tau, not along a separately sampled path.
Both greedy action selection and bootstrap evaluation use the delayed model
(plain DQN). The endpoint is reused for action selection.

These equations use ordinary DQN rewards. The paper's practical Q-only variant
scales rewards by `(1-gamma)`; this example does not apply that scaling implicitly.
It also uses discrete greedy action selection and supervises the integration grid.


In [ ]:
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": flow.parameters(), "lr": LR_DIFFUSION},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
objective = ValueDiffusionObjective(
    reward=None,
    discount=boundary_discount(
        gamma_step=GAMMA_STEP,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
)


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    modules.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        inputs = inputs.to(device)
        objective_data = objective_data.to(device)
        sequence_ids = objective_data["sequence_ids"]
        step_ids = objective_data["step_ids"]
        out = encoder(inputs)
        with torch.no_grad():
            delayed_out = delayed_encoder(inputs)
            sequences, noise_ids = torch.unique(sequence_ids[step_ids], return_inverse=True)
            z0 = torch.randn(len(sequences), device=device, dtype=torch.float32)[noise_ids]
            noise = torch.randn_like(z0)
            delayed_trajectory = delayed_flow.trajectory(delayed_out["z"], initial_samples=z0)
            z_tau = delayed_trajectory.samples
            flow_inputs = objective.inputs(
                objective_data=objective_data,
                sequence_ids=sequence_ids,
                step_ids=step_ids,
                backup_samples=z_tau,
                backup_action_predictions=delayed_trajectory.returns,
                noise_levels=delayed_trajectory.noise_levels,
                noise=noise,
            )
            delayed_predictions = delayed_flow(delayed_out["z"], z_tau, times=delayed_trajectory.noise_levels)
        predictions = flow(out["z"], flow_inputs, times=delayed_trajectory.noise_levels)
        losses, metrics = objective(
            objective_data=objective_data,
            sequence_ids=sequence_ids,
            step_ids=step_ids,
            predictions=predictions,
            backup_value_predictions=delayed_predictions,
            backup_action_predictions=delayed_trajectory.returns,
            noise=noise,
        )
        metrics["flow_inputs"] = flow_inputs
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak_update(target=delayed_encoder.embedding, source=encoder.embedding, tau=POLYAK_TAU_EMBEDDING)
        polyak_update(target=delayed_encoder.backbone, source=encoder.backbone, tau=POLYAK_TAU_BACKBONE)
        polyak_update(target=delayed_flow, source=flow, tau=POLYAK_TAU_DIFFUSION)
    assert loss is not None
    return (loss, metrics)

## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`, as in example 2.
Print velocity MSE and each flow step's loss. Training and action selection each
use one sample.


In [ ]:
try:
    for cycle in range(NUM_CYCLES):
        loss, metrics = run_train(num_steps=TRAIN_STEPS)
        print(f"cycle={cycle} train loss={loss.item():.4f}")
        print("per-flow-step losses:", metrics["per_flow_step"].tolist())
finally:
    loader.close()


## Save Locally

Save the encoder, ValueFlow model (including its context RMSNorm and flow schedule),
tokenizer, and explicit diffusion settings. Reconstruct the encoder and flow
model using the class definitions above and these settings before loading weights. Optimizers and delayed models
are not included, matching the weight-only saves in the other examples.

For action values, encode once, draw one noise value per readout, and integrate
under `torch.no_grad()`:

```python
features = encoder(inputs)["z"]
seeds = torch.randn(len(features), device=features.device)
action_values = flow.trajectory(features, initial_samples=seeds).returns
```
Notebook 15 would need this conditional flow path to use the checkpoint. This
notebook saves locally for review.


In [ ]:
modules.eval().to("cpu")
save_model(model=modules, path=OUTPUT_DIR)
save_tokenizer(tokenizer=tokenizer, path=OUTPUT_DIR)
settings = {
    "pretrained": PRETRAINED,
    "diffusion_hidden_dim": DIFFUSION_HIDDEN_DIM,
    "diffusion_network_layers": DIFFUSION_NETWORK_LAYERS,
    "diffusion_output_scale": DIFFUSION_OUTPUT_SCALE,
    "diffusion_steps": DIFFUSION_STEPS,
    "max_actions": MAX_ACTIONS,
    "diffusion_return_num_frequencies": DIFFUSION_RETURN_NUM_FREQUENCIES,
    "diffusion_time_num_frequencies": DIFFUSION_TIME_NUM_FREQUENCIES,
    "gamma_step": GAMMA_STEP,
}
(OUTPUT_DIR / "value_diffusion_settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print(f"Saved encoder, ValueFlow model, settings, and tokenizer locally in {OUTPUT_DIR}")
